# IDF: consistency constraints for the coupling, the models solve themselves

IDF (individual discipline feasible) sits between MDF and SAND. The optimiser owns the
iteration variables and the coupling variables. The coupling variables are the copies
that open the feedback loops between models, and the optimiser answers one consistency
equality per copy, `g(x) - x = 0`, next to the file's own constraints. Solves that live
inside a single model are not lifted: the coil width root find, the ion-temperature
iteration and the power-conversion step each stay their own problem, solved inside every
optimiser evaluation. So each model is internally consistent at every iterate. The
models agree with each other only at the optimum.

IDF takes four operations on the graph. Cut the feedback loops. Insert the optimisation
problem. `Residualise` and `Combine` only the problems the cut created, which are the
coupling between models. `Nest` the problems the models contain themselves inside it. SAND
residualises and combines all of them; MDF nests all of them. On this file the optimiser
gets three coupling unknowns and three consistency equalities on top of the eight
iteration variables and fourteen constraints.

In [1]:
import os, sys, time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
# The editable cottax checkout beside this repo (`../jaxgraph/src`), ahead of any
# installed copy.
_jaxgraph = REPO.parent.parent / "jaxgraph" / "src"
for entry in (str(REPO), str(_jaxgraph)):
    if Path(entry).is_dir() and entry not in sys.path:
        sys.path.insert(0, entry)

import jax
jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
import cottax
import numpy as np


print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## The graph

The models of the Helias stellarator input file, as the port declares them. Each node
is one model function; it reads and writes PROCESS's own variables (`.physics.rmajor`
is `data.physics.rmajor`). One disconnected node is left out. The file also poses the
problem: iteration variables `ixc`, constraints `icc`, figure of merit.
`configurations/stellarator_helias.py` states all of that as a tree -- the input
file converted once -- and `native.reference_of` reads it without running PROCESS.

In [2]:
CONFIGURATION = "stellarator_helias"                   # configurations/stellarator_helias.py: the machine, its values, its problem

import re

from cottax.answerable import AnswerableGraph
from cottax.plan import Plan
from cottax.problem import is_fixed_point

from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.cottax.input import native
from functional_process.configurations import load
from functional_process.cottax.input.indat import graph_for
from functional_process.cottax.architectures.evaluate import without_excluded
from functional_process.cottax.queries import declared
from functional_process.cottax.visualization.grouping import driver_name, problem_kind

configuration = load(CONFIGURATION)
ref = native.reference_of(configuration)      # ixc, icc, bounds, cold values -- PROCESS-free
sw = configuration.problem.switches           # the static switch values the condition nodes are bound with
machine_graph = graph_for(configuration.machine)
raw = without_excluded(machine_graph)

print(f"{len(raw.nodes)} nodes; cyclic components of sizes {[len(c) for c in raw.graph.cycles]}")
print("problems the models declare themselves:")
for p in declared(raw):
    print(f"   {p.spelling:55s} {problem_kind(raw[p])}")
print(f"\nixc = {ref.ixc}")
print(f"icc = {ref.icc}  ({ref.n_equality} equalities), objective: figure of merit {ref.i_figure_merit}")

152 nodes; cyclic components of sizes [2, 6, 2, 2, 2]
problems the models declare themselves:
   ^problem.stellarator.coils.intersect                    root-find
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point
   ^problem.power.delta_eta_step                           fixed-point

ixc = [2, 3, 4, 6, 10, 56, 59, 109]
icc = [2, 16, 24, 8, 17, 18, 67, 82, 83, 62, 32, 34, 35, 65]  (2 equalities), objective: figure of merit 6


## The recipe

### Step 1: cut the feedback loops

Each group of models that feed back into each other is opened with copies. `mda.CUTS`
names nine variables, picked by hand so that one iteration of the copies is one PROCESS
pass. `mda.cut_ops` turns the ones present in this graph into one operation per loop.
Each `FixedPointCut` gives the readers of a variable a copy, `^hat.x`, and adds the
requirement that the copy equals the computed value. That requirement is a fixed-point
problem. The `mda_gauss_seidel` notebook derives such cuts mechanically, and a recipe
from there can be used here instead.

In [3]:
from functional_process.cottax.architectures.mda import cut_graph, cut_ops

plan = Plan(raw)
for op in cut_ops(raw):
    plan = plan + op
print_recipe(plan)
print("\nsame as mda.cut_graph(raw):", plan.graph == cut_graph(raw))
minted = [p for p in declared(plan.graph) if p not in set(declared(raw))]
print("problems the cut minted:", [p.spelling for p in minted])

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)

same as mda.cut_graph(raw): True
problems the cut minted: ['^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']


### Step 2: state the file's problem

One node per active constraint and one for the figure of merit. They compute the same
values `constraints.py` and `objectives.py` do. Then the optimisation problem itself,
an `Optimise` node: minimise the objective over the iteration variables, subject to the
constraints. All are inserted into the graph like any other node. `sand.optimise_graph`
does the same; it is spelled out here so the step is visible.

In [4]:
from cottax.names import PathMap
from cottax.plan import Insert
from cottax.problem import Optimise
from cottax.spec import NodePath
from jax.tree_util import GetAttrKey

from functional_process.cottax.input.indat import objective_selection
from functional_process.cottax.architectures.sand import constraint_nodes, iteration_variable_path, objective_nodes, optimise_graph

nodes, equalities, inequalities, omitted = constraint_nodes(plan.graph, ref.icc, ref.n_equality, sw)
objective_built, objective = objective_nodes(plan.graph, objective_selection(ref.i_figure_merit), sw)
nodes.update(objective_built)
OPT = NodePath((GetAttrKey("Opt"),))
nodes[OPT] = Optimise(
    objective=objective,
    unknowns=tuple(iteration_variable_path(i) for i in ref.ixc),
    equalities=tuple(equalities),
    inequalities=tuple(inequalities),
)
plan = plan + Insert(PathMap(nodes.items()))
print_recipe(plan)
if omitted:
    print("constraints this port cannot state yet:", omitted)
reference_graph, _, _ = optimise_graph(cut_graph(raw), ref.ixc, ref.icc, ref.n_equality, ref.i_figure_merit, switch_values=sw)
print("same as sand.optimise_graph(...):", plan.graph == reference_graph)

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
same as sand.optimise_graph(...): True


The optimisation problem reads the objective and the constraints. It also owns the
iteration variables, which almost everything reads. So it closes one big loop over most
of the machine. That loop now contains several problems. The graph is refused as
answerable until it is told how they relate, and the refusal names the two options:
`Combine` and `Nest`. Choosing between them is choosing the architecture.

In [5]:
big = max(plan.graph.graph.components, key=len)     # the components, with no claim that they can be run
print(f"largest block: {len(big)} of {len(plan.graph.nodes)} nodes, holding",
      [p.spelling for p in declared(plan.graph) if p in set(big)], "\n")
try:
    AnswerableGraph(plan.graph)                       # the proof that a schedule could be built on it
except ValueError as refusal:                 # the node list dropped from the message
    print(re.sub(r"cycle \(.*?\) declares", "the cycle declares", str(refusal), flags=re.S))

largest block: 123 of 170 nodes, holding ['^problem.stellarator.coils.intersect', '^problem.physics.profiles.ion_vol_avg_temperature', '^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single', '.Opt'] 



the block declares several problems ((NodePath(^problem.stellarator.coils.intersect), NodePath(^problem.physics.profiles.ion_vol_avg_temperature), NodePath(^problem.physics.proton_rate_density.cycle), NodePath(^problem.fwbs.f_ster_div_single), NodePath(.Opt))) with nothing saying which is outer -- one driver answers one problem, so `Combine` them into a single problem over every unknown, or `NestInside` one of them. Which is a modelling decision, not something the blocking can read off the graph


### Step 3: residualise and merge the coupling, nest the rest

The problems the cut created are the coupling between models. The problems the models
declared are internal to one model. Only the first group is folded into the optimiser:
residualise, then merge into `^problem.idf`. The second group is nested inside it.
`idf.idf_graph` builds the same graph in one call.

In [6]:
from cottax.rewrites import Combine, Nest, Residualise
from cottax.visualization.xdsm import problems_at

from functional_process.cottax.architectures import idf
from functional_process.cottax.architectures.evaluate import mda_env

driven, env = mda_env(ref, graph=machine_graph)          # one converged MDA on the cut graph, to seed from

coupling = [p for p in minted if is_fixed_point(plan.graph[p])]
internal = [p for p in declared(raw)]
print("coupling (folded into the optimiser):", [p.spelling for p in coupling])
print("internal (nested inside it):        ", [p.spelling for p in internal], "\n")

for p in coupling:
    plan = plan + Residualise(p)                         # `x = g(^hat.x)`  ->  `g(^hat.x) - x = 0`
combine = Combine(NodePath((GetAttrKey("idf"),)), (OPT, *coupling))   # the optimiser first: its unknowns lead
plan = plan + combine
IDF = combine.problem                                    # the combined problem's minted name, `^problem.idf`
component = next(c for c in plan.graph.graph.components if IDF in c)
for name in plan.graph.subgraph(component).outermost_problems:
    if name != IDF:
        plan = plan + Nest(name, IDF)                    # every other problem on the cycle: inside
graph = plan.graph

print("the whole recipe:")
print_recipe(plan)

reference_graph, _, _ = idf.idf_graph(machine_graph, ref.ixc, ref.icc, ref.n_equality, ref.i_figure_merit, switch_values=sw)
print("\nsame as idf.idf_graph(...):", graph == reference_graph)
blocking = AnswerableGraph(graph)                        # proved: one problem per level
print("\nanswered at the top level:", [p.spelling for p in problems_at(graph) if p is not None])
print("nested inside the optimiser's block:", [p.spelling for p in problems_at(graph.interior(IDF)) if p is not None])

coupling (folded into the optimiser): ['^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']
internal (nested inside it):         ['^problem.stellarator.coils.intersect', '^problem.physics.profiles.ion_vol_avg_temperature', '^problem.power.delta_eta_step'] 

the whole recipe:
   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
   residualise(^problem.physics.proton_rate_density.cycle)
   residualise(^problem.fwbs.f_ster_div_single)
   combine(^problem.idf <- .Opt, ^problem.physics.proton_rate_density.cycle, ^problem.fwbs.f_ster_div_single)
   nest_inside(^problem.idf)

answered at the top level: ['^problem.idf', '^problem.power.delta_eta_step']
nested inside the optimiser's

### Assign the drivers

VMCON on the merged problem. Fixed-point iteration and Newton on the nested ones; they
converge inside every VMCON evaluation and are differentiated through.

In [7]:
from functional_process.cottax.architectures.sand import sand_schedule, sand_shape

schedule = sand_schedule(graph, None, bounds=ref.bounds)        # `Assign` VMCON on `^problem.idf`, Picard/Newton inside
shape = sand_shape(schedule)
drive = shape["drive"]
print(f"the IDF block: {shape['drive_nodes']} nodes, {shape['unknowns']} unknowns "
      f"({shape['design']} design), {shape['conditions']} conditions "
      f"({shape['equalities']} equalities, {shape['inequalities']} inequalities); "
      f"{shape['schedule_steps']} schedule steps in all")
print("unknowns:", [u.spelling for u in drive.unknowns])
runnable = schedule.answerable.graph
for p in declared(runnable):
    print(f"   {p.spelling:55s} {problem_kind(runnable[p]):12s} {driver_name(runnable[p])}")

the IDF block: 123 nodes, 11 unknowns (11 design), 18 conditions (5 equalities, 12 inequalities); 47 schedule steps in all
unknowns: ['.physics.b_plasma_toroidal_on_axis', '.physics.rmajor', '.physics.temp_plasma_electron_vol_avg_kev', '.physics.nd_plasma_electrons_vol_avg', '.physics.hfact', '.tfcoil.t_tf_superconductor_quench', '.tfcoil.f_a_tf_turn_cable_copper', '.physics.f_nd_alpha_thermal_electron', '^hat.physics.proton_rate_density', '^hat.physics.fusden_alpha_total', '^hat.fwbs.f_ster_div_single']
   ^problem.stellarator.coils.intersect                    root-find    SeededNewtonDriver
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point  PicardDriver
   ^problem.power.delta_eta_step                           fixed-point  PicardDriver
   ^problem.idf                                            combined     VmconDriver


## The process

The DSM in run order. The optimiser's box surrounds the coupled block, and inside it are
the smaller boxes of the model-internal solves. MDF shows a box around every iterated
group and SAND shows none inside; IDF shows exactly the model-internal ones. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [8]:
from functional_process.cottax.visualization.render_xdsm import SPELLING
from functional_process.cottax.visualization.grouping import render_grouped_dsm_html, structure_order

blocking = schedule.answerable
dsm = render_grouped_dsm_html(
    blocking, order=structure_order(blocking),
    title="stellarator_helias -- IDF: coupling lifted into VMCON, discipline solves nested inside",
    file_name="dsm_idf", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch


written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/idf/dsm_idf.html


## Run it

As in the SAND notebook, with one addition the nesting needs. The starting values of the
nested solves are not unknowns of the optimiser's block, so the standard seeding leaves
them at the input file's cold values. `mda.seed_starts` takes them from the converged
analysis instead. The two-driver case needed the same.

In [9]:
from functional_process.cottax.architectures import mdf
from functional_process.cottax.architectures.drivers import Status
from functional_process.cottax.architectures.mda import seed_starts
from functional_process.cottax.architectures.session import recorder, trace_tail
from functional_process.cottax.architectures.evaluate import inputs_only, seed_block
from functional_process.cottax.architectures.session import SAND_MAX_ITER
from functional_process.cottax.architectures.sand import residual_condition_scales
from functional_process.cottax.architectures.evaluate import run_schedule

trace = []
solve = sand_schedule(graph, None, bounds=ref.bounds,
                      condition_scale=residual_condition_scales(drive, env),
                      callback=recorder(trace), max_iter=SAND_MAX_ITER)
solve_drive = sand_shape(solve)["drive"]
design_vars = {iteration_variable_path(i) for i in ref.ixc}
seeded, borrowed = seed_block(solve, solve_drive, ref.cold, env, design=design_vars)
starts = seed_starts(solve, env, exclude=design_vars)
seeded.update(starts)
print("start ports re-seeded from the MDA:", [v.spelling for v in starts], "\n")

began = time.perf_counter()
out = run_schedule(solve, inputs_only(solve, seeded), whole=False)
print(f"solved in {time.perf_counter() - began:.1f} s (first solve: includes compilation)")

status = int(np.asarray(mdf.verdict(out, Status, solve_drive.problem)))
iterations, objf, max_eq, min_ie = trace_tail(trace)
print(f"VMCON: status {status}, {iterations} iterations")
print(f"objective {objf:.8f}   max|eq| {max_eq:.1e}   min ineq {min_ie:+.1e}")
print("design:", {i: round(float(np.asarray(out[iteration_variable_path(i)])), 4) for i in ref.ixc})

trace.clear()
began = time.perf_counter()
out = run_schedule(solve, inputs_only(solve, seeded), whole=False)
print(f"\nwarm solve: {time.perf_counter() - began:.2f} s, {len(trace)} iterations")

start ports re-seeded from the MDA: ['^guess.stellarator.wp_width_r_min', '^guess.physics.temp_plasma_ion_vol_avg_kev', '^guess.power.delta_eta', '^guess.physics.proton_rate_density', '^guess.physics.fusden_alpha_total', '^guess.fwbs.f_ster_div_single'] 



solved in 9.5 s (first solve: includes compilation)
VMCON: status 0, 23 iterations
objective 1.21844143   max|eq| 3.6e-11   min ineq -7.8e-10
design: {2: 4.7164, 3: 26.6445, 4: 5.7027, 6: 1.7391773832251176e+20, 10: 1.048, 56: 31.8104, 59: 0.7177, 109: 0.0299}



warm solve: 0.62 s, 23 iterations


## The three answers to one loop

| | MDF | IDF | SAND |
|---|---|---|---|
| operations | nest everything inside the optimiser | residualise and merge the coupling; nest the model-internal solves | residualise and merge everything |
| optimiser unknowns | iteration variables | + coupling copies | + coupling copies + every model-internal unknown |
| extra equalities | none | one per coupling copy | one per lifted unknown |
| per evaluation | a converged analysis | one pass, model-internal solves converged | one pass |
| models consistent | at every iterate | within each model at every iterate; between models at the optimum | at the optimum |

Same models, same optimum. Only the operations differ, and the graph records which were
applied.

In [10]:
RESULT = {"status": status, "iterations": iterations, "objf": objf, "max_eq": max_eq, "min_ie": min_ie,
          "design": {i: float(np.asarray(out[iteration_variable_path(i)])) for i in ref.ixc},
          "unknowns": len(solve_drive.unknowns), "conditions": len(solve_drive.conditions),
          "nested": [p.spelling for p in problems_at(graph.interior(IDF)) if p is not None]}
RESULT

{'status': 0,
 'iterations': 23,
 'objf': 1.218441432809987,
 'max_eq': 3.639666346089143e-11,
 'min_ie': -7.814808800077344e-10,
 'design': {2: 4.7164495986924,
  3: 26.644455788358055,
  4: 5.702734495286721,
  6: 1.7391773832251176e+20,
  10: 1.047952388067319,
  56: 31.81044366522202,
  59: 0.7176937077117317,
  109: 0.02992503718934735},
 'unknowns': 11,
 'conditions': 18,
 'nested': ['^problem.physics.profiles.ion_vol_avg_temperature',
  '^problem.stellarator.coils.intersect']}